<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/err/parquet_data_all_stocks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import zipfile
import shutil
import pandas as pd
import glob

# 1. Load the Original Database
# ---------------------------------------------------------
parquet_file = "/content/equities_combined.parquet"

# Load the original dataframe so we can use its SERIES column later for filtering
if os.path.exists(parquet_file):
    df = pd.read_parquet(parquet_file)
    print(f"Loaded original database with {len(df)} rows.")
else:
    raise FileNotFoundError(f"{parquet_file} not found. Please ensure the file is in the working directory.")



Loaded original database with 3754607 rows.


In [ ]:

# 2. Extract ZIP Files Extract the CSV files to 'levelone' folder
# ---------------------------------------------------------
source_folder = "/content"           # Folder containing ZIP files
levelone_folder = "/content/levelone"

os.makedirs(levelone_folder, exist_ok=True)

def extract_all_zips(source, destination):
    # First copy all zip files to levelone
    for file in os.listdir(source):
        if file.endswith(".zip"):
            shutil.copy(os.path.join(source, file), destination)

    # Now recursively extract inside levelone
    while True:
        zip_found = False
        for root, dirs, files in os.walk(destination):
            for file in files:
                if file.endswith(".zip"):
                    zip_found = True
                    zip_path = os.path.join(root, file)

                    with zipfile.ZipFile(zip_path, 'r') as z:
                        z.extractall(root)

                    os.remove(zip_path)
                    print(f"Extracted and removed: {file}")

        if not zip_found:
            break

    print("All ZIP files fully extracted into levelone.")

extract_all_zips(source_folder, levelone_folder)



In [4]:
# 3. Read and append all new CSV files into a temporary dataframe
# ---------------------------------------------------------
# 1. Get all CSV files
csv_files = glob.glob(os.path.join(levelone_folder, "*.csv"))

# 2. Filter out any files that have "PE" in their filename
csv_files = [f for f in csv_files if "PE" not in os.path.basename(f)]

df_list = []

for file in csv_files:
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

full_data_from_csv = pd.concat(df_list, ignore_index=True)



In [5]:
# 4. Column Mapping
# ---------------------------------------------------------
# Ensures the Pandas Dataframe and the original database have the same Column Names
column_mapping = {
    "TckrSymb": "SYMBOL",
    "SctySrs": "SERIES",
    "OpnPric": "OPEN",
    "HghPric": "HIGH",
    "LwPric": "LOW",
    "ClsPric": "CLOSE",
    "LastPric": "LAST",
    "PrvsClsgPric": "PREVCLOSE",
    "TtlTradgVol": "TOTTRDQTY",
    "TtlTrfVal": "TOTTRDVAL",
    "BizDt": "TIMESTAMP",
    "TtlNbOfTxsExctd": "TOTALTRADES",
    "ISIN": "ISIN"
}

# Subset + rename
subset_full_data = full_data_from_csv[list(column_mapping.keys())].copy()
subset_full_data.rename(columns=column_mapping, inplace=True)



In [7]:
# 5. Filter the new data based on SERIES in the original database
# ---------------------------------------------------------
# Extract unique series from the original database
valid_series = df["SERIES"].unique()

# Keep only rows in the new data where the SERIES exists in the original database
subset_full_data = subset_full_data[subset_full_data["SERIES"].isin(valid_series)]


print("Valid Series" , valid_series)
print("-" * 30)

Valid Series ['YR' 'YW' 'N1' 'N3' 'N0' 'NX' 'NV' 'NC' 'N4' 'Z4' 'ZQ' 'NO' 'NF' 'NR'
 'AL' 'AZ' 'YY' 'ZL' 'GS' 'NT' 'Z8' 'ZT' 'Y3' 'N5' 'N8' 'NG' 'NJ' 'NM'
 'N6' None 'NE' 'Y5' 'Z2' 'ND' 'N7' 'NU' 'ZP' 'NH' 'NP' 'NB' 'NS' 'N2'
 'ZR' 'YK' 'Z6' 'N9' 'BV' 'TB' 'EQ' 'BE' 'BL' 'IT' 'SG' 'NL' 'NN' 'NI'
 'NK' 'YX' 'YM' 'YH' 'NW' 'NZ' 'YP' 'Y8' 'ZE' 'NQ' 'YI' 'ZF' 'ZK' 'YT'
 'YC' 'YZ' 'ZG' 'Z3' 'ZY' 'BZ' 'AW' 'ZI' 'ZB' 'Z9' 'YV' 'Z7' 'ZW' 'AO'
 'AC' 'AR' 'YQ' 'ZD' 'YA' 'BS' 'ZM' 'Z5' 'ZH' 'ZN' 'AI' 'AX' 'ZJ' 'YL'
 'AP' 'ZU' 'BX' 'Y2' 'AG' 'AB' 'Y1' 'BF' 'YO' 'AK' 'AN' 'BC' 'Z0' 'Y0'
 'YS' 'YG' 'ZX' 'AJ' 'AH' 'AV' 'Y7' 'ST' 'SM' 'E1' 'P1' 'W1' 'MF' 'T0'
 'X1' 'IV' 'SZ' 'SF' 'BO' 'RR' 'NY' 'Y4' 'W3' 'ZS' 'Y6' 'BI' 'YN' 'AT'
 'AY' 'YU' 'ZC' 'ZZ' 'AA' 'BG' 'BJ' 'BM' 'E3' 'ID' 'P2' 'Y9' 'GB' 'ZA'
 'YB' 'YJ' 'Z1' 'DR' 'SO']
------------------------------


In [8]:
df.tail(2)

,SYMBOL,SERIES,OPEN,HIGH,LOW,CLOSE,LAST,PREVCLOSE,TOTTRDQTY,TOTTRDVAL,TIMESTAMP,TOTALTRADES,ISIN
3754605,ZYDUSWELL,EQ,379.25,385.9,375.1,383.0,383.00,377.1,77580,29456614.3,2026-03-05,8041,INE768C01028
3754606,ZYDUSWELL,EQ,381.00,386.6,380.0,381.2,380.55,383.0,67849,25972773.7,2026-03-06,7033,INE768C01028


In [9]:
# 6. Format and Combine
# ---------------------------------------------------------
# Standardize dates
df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])
subset_full_data["TIMESTAMP"] = pd.to_datetime(subset_full_data["TIMESTAMP"])

# Append new filtered rows to the original dataframe
combined = pd.concat([df, subset_full_data], ignore_index=True)

# Sort chronologically by Symbol, then Date
combined = combined.sort_values(["SYMBOL", "TIMESTAMP"])

# Reset index for a clean dataframe
combined = combined.reset_index(drop=True)

# Check consistency
print("-" * 30)
print("Rows in original database:", len(df))
print("Rows in new (filtered) data:", len(subset_full_data))
print("Total rows in combined dataset:", len(combined))
print("Latest Date:", combined["TIMESTAMP"].max())
print("Earliest Date:", combined["TIMESTAMP"].min())
print("-" * 30)



------------------------------
Rows in original database: 3754607
Rows in new (filtered) data: 203377
Total rows in combined dataset: 3957984
Latest Date: 2026-06-30 00:00:00
Earliest Date: 2020-01-01 00:00:00
------------------------------


In [10]:
combined.tail(2)

,SYMBOL,SERIES,OPEN,HIGH,LOW,CLOSE,LAST,PREVCLOSE,TOTTRDQTY,TOTTRDVAL,TIMESTAMP,TOTALTRADES,ISIN
3957982,ZYDUSWELL,EQ,522.0,577.5,518.15,569.45,565.4,518.10,15733500,8.812249e+09,2026-06-29,235174,INE768C01028
3957983,ZYDUSWELL,EQ,567.9,580.0,558.90,575.40,575.0,569.45,2239650,1.281350e+09,2026-06-30,42846,INE768C01028


In [11]:
# 7. Save to Parquet
# ---------------------------------------------------------
combined.to_parquet(parquet_file, index=False)
print(f"Successfully saved updated data to {parquet_file}")

Successfully saved updated data to /content/equities_combined.parquet
